#1: Imports and Path Setup

In [ ]:
import os
import sys
import numpy as np
import tensorflow as tf

In [ ]:
PROJECT_ROOT = '/content/drive/MyDrive/Transformer_Mine'

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
%cd /content/drive/MyDrive/Transformer_Mine

Mounted at /content/drive
/content/drive/MyDrive/Transformer_Mine


In [ ]:
from encoder import Encoder
from decoder import Decoder
from data_pipeline import (
    create_padding_mask,
    create_decoder_mask,
)

print("TensorFlow:", tf.__version__)
print("Project root:", PROJECT_ROOT)

TensorFlow: 2.20.0
Project root: /content/drive/MyDrive/Transformer_Mine


# 2: Mathematical Foundation

## Complete Transformer

From the paper (Section 3):

```text
src_ids
   |
Encoder
   |
enc_output: (batch, src_len, d_model)
   |
   +-------> Decoder <------- tgt_ids
                 |
            dec_output: (batch, tgt_len, d_model)
                 |
            Linear projection: (batch, tgt_len, vocab_size)
                 |
            logits
            (no softmax — applied in loss function)
```

### Final Linear Projection

The final linear layer projects:

$$
d_{\text{model}} \rightarrow \text{vocab\_size}
$$

Therefore, the output shape is:

```text
(batch, tgt_len, vocab_size)
```

Each target position produces a score for every token in the vocabulary.

### Weight Tying

From the paper, Section 3.4:

> "We share the same weight matrix between the two embedding layers and the pre-softmax linear transformation."

The weight-sharing relationship is:

```text
encoder_embedding.weights
          ==
decoder_embedding.weights
          ==
output_projection.weightsᵀ
```

This technique is known as **weight tying**.

It significantly reduces the number of trainable parameters and was used in the original Transformer implementation.

### Training vs. Inference

**During training:**

```text
logits
   ↓
Cross-Entropy Loss
```

The model outputs raw logits, and the cross-entropy loss operates directly on them. A separate softmax operation is therefore not required before computing the loss.

**During inference:**

```text
logits
   ↓
softmax
   ↓
probabilities
   ↓
argmax / decoding
   ↓
predicted token
```

### Output Shape

The final decoder output is projected to vocabulary-sized logits:

```text
dec_output
    : (batch, tgt_len, d_model)

        ↓

Linear Projection
    : d_model → vocab_size

        ↓

logits
    : (batch, tgt_len, vocab_size)
```

Each position predicts a probability distribution over the entire target vocabulary.


# 3: Transformer Model

In [ ]:
class Transformer(tf.keras.Model):
    """
    Complete Transformer model for sequence-to-sequence translation.

    Implements the Transformer Base configuration from:
    Vaswani et al. 2017 "Attention Is All You Need"

    Args:
        num_layers : encoder and decoder layers (paper: 6)
        d_model : model dimension (paper: 512)
        num_heads : attention heads (paper: 8)
        d_ff : FFN inner dimension (paper: 2048)
        src_vocab : source vocabulary size
        tgt_vocab : target vocabulary size
        max_len : max sequence length (paper: 5000)
        dropout : dropout rate (paper: 0.1)
        tie_weights : share embedding and output projection weights
    """

    def __init__(self, num_layers, d_model, num_heads, d_ff, src_vocab, tgt_vocab, max_len=5000, dropout=0.1, tie_weights=True, **kwargs,):
        super().__init__(**kwargs)

        self.num_layers = num_layers
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_ff = d_ff
        self.src_vocab = src_vocab
        self.tgt_vocab = tgt_vocab
        self.tie_weights = tie_weights

        # Encoder stack
        self.encoder = Encoder(
            num_layers=num_layers,
            d_model=d_model,
            num_heads=num_heads,
            d_ff=d_ff,
            vocab_size=src_vocab,
            max_len=max_len,
            dropout=dropout,
            name = 'encoder',
        )

        # Decoder stack
        self.decoder = Decoder(
            num_layers=num_layers,
            d_model=d_model,
            num_heads=num_heads,
            d_ff=d_ff,
            vocab_size=tgt_vocab,
            max_len=max_len,
            dropout=dropout,
            name = 'decoder',
        )

        # Final linear projection: d_model -> tgt_vocab
        # No activation — softmax applied in loss or decoding
        self.output_projection = tf.keras.layers.Dense(
            tgt_vocab,
            use_bias=False,
            name='output_projection',
        )

    def call(self, src_ids, tgt_ids, training=False, enc_mask=None, dec_mask=None,):
        """
        Forward pass of the complete Transformer.

        Args:
            src_ids : source token IDs (batch, src_len)
            tgt_ids : target token IDs (batch, tgt_len)
                       during training: decoder input [BOS, t1, t2, ...]
            training : controls dropout
            enc_mask : encoder padding mask (auto-generated if None)
            dec_mask : decoder combined mask (auto-generated if None)

        Returns:
            logits : (batch, tgt_len, tgt_vocab)
            attention_weights : dict of all encoder and decoder attn weights
        """

        # Auto-generate masks if not provided
        if enc_mask is None:
            enc_mask = create_padding_mask(src_ids)

        if dec_mask is None:
            tgt_len = tf.shape(tgt_ids)[1]
            dec_mask = create_decoder_mask(tgt_len)


        # Encoder forward pass
        enc_output, enc_attn = self.encoder(
            src_ids,
            mask = enc_mask,
            training = training,
        )

        # Decoder forward pass
        dec_output, dec_attn = self.decoder(
            tgt_ids,
            enc_output,
            dec_mask = dec_mask,
            enc_mask = enc_mask,
            training = training,
        )

        # Final linear projection to vocabulary
        logits = self.output_projection(dec_output)

        attention_weights = {**enc_attn, **dec_attn}

        return logits, attention_weights

    def build_masks(self, src_ids, tgt_ids):
        """
        Utility to generate all required masks.

        Args:
            src_ids : (batch, src_len)
            tgt_ids : (batch, tgt_len)

        Returns:
            enc_mask : (batch, 1, 1, src_len)
            dec_mask : (batch, 1, tgt_len, tgt_len)
        """

        enc_mask = create_padding_mask(src_ids)

        dec_mask = create_decoder_mask(tgt_ids.shape[1])

        return enc_mask, dec_mask

    def get_config(self):
        config = super().get_config()

        config.update({
            'num_layers' : self.num_layers,
            'd_model' : self.d_model,
            'num_heads' : self.num_heads,
            'd_ff' : self.d_ff,
            'src_vocab' : self.src_vocab,
            'tgt_vocab' : self.tgt_vocab,
            'tie_weights' : self.tie_weights,
        })

        return config

# 4: Weight Tying

## Weight Sharing

From the paper, Section 3.4:

> The encoder embedding, decoder embedding, and output projection share the same weight matrix.

This is implemented **after the model is built**, when all required weights already exist.

### Weight-Tying Mechanism

The output projection kernel is overridden so that it points to the **decoder embedding matrix**.

```text
Encoder Embedding
        |
        |  shared weights
        ↓
Decoder Embedding
        |
        |  shared weights
        ↓
Output Projection
```

### Vocabulary Requirement

Weight tying requires:

```text
src_vocab == tgt_vocab
```

This condition is satisfied when using a **shared BPE vocabulary**, as in the original Transformer setup for the EN-DE translation task.

### Why Weight Tying Is Used

Weight tying:

* Reduces the total number of trainable parameters.
* Ensures the embedding and output projection use the same learned representation.
* Matches the parameter-sharing strategy described in **Section 3.4 of the original Transformer paper**.


In [ ]:
def apply_weight_tying(model):
    """
    Tie encoder embedding, decoder embedding, and output
    projection weights as described in paper Section 3.4.

    Requires src_vocab == tgt_vocab (shared vocabulary).

    The output projection W_out shape is (d_model, vocab_size).
    The embedding matrix E shape is (vocab_size, d_model).
    Weight tying: W_out = E^T

    This is implemented by building a custom call that uses
    the embedding weights transposed for the final projection.

    Args:
        model: a built Transformer instance

    Note:
        Full weight tying via kernel assignment is not directly
        supported in all TF versions. We document the intention
        here and implement it as a tied projection in the
        tied_output_projection method below.
    """

    enc_embed = model.encoder.embedding
    dec_embed = model.decoder.embedding

    assert enc_embed.embeddings.shape == dec_embed.embeddings.shape, \
        "Encoder and decoder vocab sizes must match for weight tying."

    print("Weight tying status:")
    print(f"Encoder embedding shape : {enc_embed.embeddings.shape}")

    print(f"Decoder embedding shape : {dec_embed.embeddings.shape}")

    print(f"Output projection shape : "
          f"{model.output_projection.kernel.shape}")

In [ ]:
class TiedTransformer(Transformer):
    """
    Transformer with weight tying between embeddings and output projection.

    Overrides the output projection to use decoder embedding weights^T.
    Requires src_vocab == tgt_vocab.
    """

    def call(self, src_ids, tgt_ids, training=False, enc_mask=None, dec_mask=None,):
        if enc_mask is None:
            enc_mask = create_padding_mask(src_ids)

        if dec_mask is None:
            dec_mask = create_decoder_mask(tgt_ids, tf.shape(tgt_ids)[1])

        enc_output, enc_attn = self.encoder(src_ids, mask=enc_mask, training = training)

        dec_output, dec_attn = self.decoder(tgt_ids, enc_output, dec_mask=dec_mask, enc_mask=enc_mask, training=training)

        # Tied projection: use decoder embedding matrix transposed
        # dec_output shape: (batch, tgt_len, d_model)
        # embedding shape : (vocab_size, d_model)
        # logits shape : (batch, tgt_len, vocab_size)
        embedding_weights = self.decoder.embedding.embeddings
        logits = tf.matmul(dec_output, embedding_weights, transpose_b=True)

        attention_weights = {**enc_attn, **dec_attn}
        return logits, attention_weights

#5: Build Transformer Base and Shape Test

In [ ]:
NUM_LAYERS = 6
D_MODEL = 512
NUM_HEADS = 8
D_FF = 2048
VOCAB_SIZE = 8000
MAX_LEN = 5000
DROPOUT = 0.1
BATCH = 2
SRC_LEN = 15
TGT_LEN = 10

In [ ]:
transformer = TiedTransformer(
    num_layers = NUM_LAYERS,
    d_model = D_MODEL,
    num_heads = NUM_HEADS,
    d_ff = D_FF,
    src_vocab = VOCAB_SIZE,
    tgt_vocab = VOCAB_SIZE,
    max_len = MAX_LEN,
    dropout = 0.0,
    tie_weights = True,
)

In [ ]:
tf.random.set_seed(42)

In [ ]:
src_ids = tf.constant([
    [5, 12, 8, 33, 7, 2, 0, 0, 0, 0, 0, 0, 0, 0, 0],
    [3, 18, 6, 44, 9, 11, 22, 4, 2, 0, 0, 0, 0, 0, 0],
], dtype=tf.int32)

tgt_ids = tf.constant([
    [1, 15, 22, 8, 2, 0, 0, 0, 0, 0],
    [1,  9,  4, 6, 3, 7, 2, 0, 0, 0],
], dtype=tf.int32)

In [ ]:
logits, attn_weights = transformer(src_ids, tgt_ids, training=False)

/usr/local/lib/python3.13/dist-packages/keras/src/layers/layer.py:982: UserWarning: Layer 'encoder_self_attention' (of type MultiHeadAttention) was passed an input with a mask attached to it. However, this layer does not support masking and will therefore destroy the mask information. Downstream layers will not see the mask.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/keras/src/layers/layer.py:982: UserWarning: Layer 'decoder_self_attention' (of type MultiHeadAttention) was passed an input with a mask attached to it. However, this layer does not support masking and will therefore destroy the mask information. Downstream layers will not see the mask.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/keras/src/layers/layer.py:982: UserWarning: Layer 'decoder_cross_attention' (of type MultiHeadAttention) was passed an input with a mask attached to it. However, this layer does not support masking and will therefore destroy the mask information. Downstream layers will no

In [ ]:

assert logits.shape == (BATCH, TGT_LEN, VOCAB_SIZE), \
    f"Logits shape wrong: {logits.shape}"

print(f"src_ids shape : {src_ids.shape}")
print(f"tgt_ids shape : {tgt_ids.shape}")
print(f"logits  shape : {logits.shape}  (batch, tgt_len, vocab_size)")
print()
print(f"logits min  : {logits.numpy().min():.4f}")
print(f"logits max  : {logits.numpy().max():.4f}")
print(f"logits mean : {logits.numpy().mean():.4f}")
print()

# Verify softmax over logits sums to 1
probs = tf.nn.softmax(logits, axis=-1)
prob_sums = tf.reduce_sum(probs, axis=-1).numpy()
print(f"softmax probabilities sum to 1:")
print(f"  min sum : {prob_sums.min():.6f}")
print(f"  max sum : {prob_sums.max():.6f}")
assert abs(prob_sums - 1.0).max() < 1e-5
print()
print("Shape test passed.")

src_ids shape : (2, 15)
tgt_ids shape : (2, 10)
logits  shape : (2, 10, 8000)  (batch, tgt_len, vocab_size)

logits min  : -2.6037
logits max  : 2.8074
logits mean : 0.0074

softmax probabilities sum to 1:
  min sum : 1.000000
  max sum : 1.000000

Shape test passed.


# 6: Parameter Count

## Parameter Count

The paper reports approximately:

```text
~65M parameters
```

for the **Transformer Base** configuration.

### Purpose

The parameter count is used as a sanity check to verify that the implementation is approximately consistent with the architecture described in the paper.


In [ ]:
total_params = sum(tf.size(w).numpy() for w in transformer.trainable_variables)
non_trainable = sum(tf.size(w).numpy() for w in transformer.non_trainable_variables)

In [ ]:
print(f"\nTrainable parameters   : {total_params:,}")
print(f"Non-trainable params   : {non_trainable:,}")


Trainable parameters   : 52,293,632
Non-trainable params   : 0


In [ ]:
enc_params = sum(tf.size(w).numpy() for w in transformer.encoder.trainable_variables)
dec_params = sum(tf.size(w).numpy() for w in transformer.decoder.trainable_variables)

In [ ]:
print(f"Encoder parameters : {enc_params:,}")
print(f"Decoder parameters : {dec_params:,}")

Encoder parameters : 22,998,016
Decoder parameters : 29,295,616


In [ ]:
print("Comparison with paper:")
print(f" Our vocab size : {VOCAB_SIZE:,}  (Stage B — 8k)")
print(f" Paper vocab size : ~37,000  (WMT14 shared BPE)")
print(f" Our total params : {total_params:,}")
print(f" Paper total params : ~65,000,000  (with vocab_size=37k)")
print("Note: parameter count scales with vocab_size via embeddings.")
print("With vocab_size=37000 our architecture will match the paper.")


Comparison with paper:
 Our vocab size : 8,000  (Stage B — 8k)
 Paper vocab size : ~37,000  (WMT14 shared BPE)
 Our total params : 52,293,632
 Paper total params : ~65,000,000  (with vocab_size=37k)
Note: parameter count scales with vocab_size via embeddings.
With vocab_size=37000 our architecture will match the paper.


In [ ]:
print("\nPer encoder layer breakdown:")
mha = 4 * D_MODEL * D_MODEL
ffn = 2 * D_MODEL * D_FF + D_FF + D_MODEL
ln = 4 * D_MODEL
print(f"MHA : {mha:>10,}")
print(f"FFN : {ffn:>10,}")
print(f"LayerNorm : {ln:>10,}")
print(f"Total : {mha+ffn+ln:>10,}")


Per encoder layer breakdown:
MHA :  1,048,576
FFN :  2,099,712
LayerNorm :      2,048
Total :  3,150,336


# 7: Model Summary

In [38]:
rows = [
    ("Architecture", "Transformer Base"),
    ("Encoder layers", str(NUM_LAYERS)),
    ("Decoder layers", str(NUM_LAYERS)),
    ("d_model", str(D_MODEL)),
    ("num_heads", str(NUM_HEADS)),
    ("d_k = d_v", str(D_MODEL // NUM_HEADS)),
    ("d_ff", str(D_FF)),
    ("dropout", str(DROPOUT)),
    ("max_len", str(MAX_LEN)),
    ("vocab_size", f"{VOCAB_SIZE:,}  (Stage B)"),
    ("Weight tying", "yes"),
    ("Positional enc", "sinusoidal (fixed)"),
    ("Attention type", "scaled dot-product"),
    ("Normalization", "POST-LN (original paper)"),
    ("Trainable params", f"{total_params:,}"),
]

In [41]:
for key, val in rows:
    print(f"{key:<22} : {val}")

print("Output shape: (batch, tgt_len, vocab_size)")
print("Each position produces logits over the full vocabulary.")
print("Softmax applied at inference; cross-entropy loss at training.")

Architecture           : Transformer Base
Encoder layers         : 6
Decoder layers         : 6
d_model                : 512
num_heads              : 8
d_k = d_v              : 64
d_ff                   : 2048
dropout                : 0.1
max_len                : 5000
vocab_size             : 8,000  (Stage B)
Weight tying           : yes
Positional enc         : sinusoidal (fixed)
Attention type         : scaled dot-product
Normalization          : POST-LN (original paper)
Trainable params       : 52,293,632
Output shape: (batch, tgt_len, vocab_size)
Each position produces logits over the full vocabulary.
Softmax applied at inference; cross-entropy loss at training.


#8: Gradient Flow Test

In [42]:
tf.random.set_seed(42)

In [43]:
transformer_grad = TiedTransformer(
    num_layers=2, d_model=D_MODEL, num_heads=NUM_HEADS,
    d_ff=D_FF, src_vocab=VOCAB_SIZE, tgt_vocab=VOCAB_SIZE,
    dropout=0.0,
)

In [44]:
src_g = tf.constant([[5, 12, 8, 33, 7, 0, 0, 0]], dtype=tf.int32)
tgt_g = tf.constant([[1, 15, 22, 8, 2, 0, 0, 0]], dtype=tf.int32)

In [45]:
with tf.GradientTape() as tape:
    logits_g, _ = transformer_grad(src_g, tgt_g, training=True)
    loss_g = tf.reduce_sum(logits_g)

grads = tape.gradient(loss_g, transformer_grad.trainable_variables)

none_grads = [v.name for v, g in zip(transformer_grad.trainable_variables, grads)
              if g is None]
nan_grads = [v.name for v, g in zip(transformer_grad.trainable_variables, grads)
              if g is not None and tf.reduce_any(tf.math.is_nan(g)).numpy()]

print(f"Total parameter tensors : {len(transformer_grad.trainable_variables)}")
print(f"None gradients : {len(none_grads)}")
print(f"NaN  gradients : {len(nan_grads)}")

if none_grads:
    print("\nParameters with None gradient:")
    for name in none_grads:
        print(f"  {name}")

if nan_grads:
    print("\nParameters with NaN gradient:")
    for name in nan_grads:
        print(f"  {name}")

if not none_grads and not nan_grads:
    print("\nGradient flow test passed.")
    print("All parameters in the complete Transformer receive valid gradients.")
else:
    print("\nGradient flow test FAILED.")

/usr/local/lib/python3.13/dist-packages/keras/src/layers/layer.py:424: UserWarning: `build()` was called on layer 'tied_transformer_3', however the layer does not have a `build()` method implemented and it looks like it has unbuilt state. This will cause the layer to be marked as built, despite not being actually built, which may cause failures down the line. Make sure to implement a proper `build()` method.
  warnings.warn(


Total parameter tensors : 62
None gradients : 0
NaN  gradients : 0

Gradient flow test passed.
All parameters in the complete Transformer receive valid gradients.


#9: Save Transformer to Drive

In [46]:
TRANSFORMER_PY = os.path.join(PROJECT_ROOT, 'transformer_model.py')

In [51]:
code = '''"""
transformer_model.py - Complete Transformer Model
Transformer Reproduction Project - Phase 10

Usage:
    from google.colab import drive
    drive.mount('/content/drive')
    %cd /content/drive/MyDrive/Transformer_Mine
    from transformer_model import Transformer, TiedTransformer
"""

import tensorflow as tf
from encoder import Encoder
from decoder import Decoder
from data_pipeline import create_padding_mask, create_decoder_mask


class Transformer(tf.keras.Model):
    """
    Complete Transformer for sequence-to-sequence translation.
    Vaswani et al. 2017 "Attention Is All You Need"
    """

    def __init__(self, num_layers, d_model, num_heads, d_ff,
                 src_vocab, tgt_vocab, max_len=5000, dropout=0.1,
                 tie_weights=True, **kwargs):
        super().__init__(**kwargs)
        self.num_layers = num_layers
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_ff = d_ff
        self.src_vocab = src_vocab
        self.tgt_vocab = tgt_vocab
        self.tie_weights = tie_weights

        self.encoder = Encoder(num_layers, d_model, num_heads, d_ff,
                               src_vocab, max_len, dropout, name="encoder")
        self.decoder = Decoder(num_layers, d_model, num_heads, d_ff,
                               tgt_vocab, max_len, dropout, name="decoder")
        self.output_projection = tf.keras.layers.Dense(tgt_vocab,
                                                        use_bias=False,
                                                        name="output_projection")

    def call(self, src_ids, tgt_ids, training=False,
             enc_mask=None, dec_mask=None):
        if enc_mask is None:
            enc_mask = create_padding_mask(src_ids)
        if dec_mask is None:
            dec_mask = create_decoder_mask(tgt_ids, tgt_len=tf.shape(tgt_ids)[1])
        enc_output, enc_attn = self.encoder(src_ids, mask=enc_mask, training=training)
        dec_output, dec_attn = self.decoder(tgt_ids, enc_output,
                                             dec_mask=dec_mask, enc_mask=enc_mask,
                                             training=training)
        logits = self.output_projection(dec_output)
        return logits, {**enc_attn, **dec_attn}

    def build_masks(self, src_ids, tgt_ids):
        enc_mask = create_padding_mask(src_ids)
        dec_mask = create_decoder_mask(tgt_ids, tgt_len=tf.shape(tgt_ids)[1])
        return enc_mask, dec_mask

    def get_config(self):
        config = super().get_config()
        config.update({"num_layers": self.num_layers, "d_model": self.d_model,
                        "num_heads": self.num_heads, "d_ff": self.d_ff,
                        "src_vocab": self.src_vocab, "tgt_vocab": self.tgt_vocab,
                        "tie_weights": self.tie_weights})
        return config


class TiedTransformer(Transformer):
    """
    Transformer with weight tying (paper Section 3.4).
    Output projection uses decoder embedding weights transposed.
    Requires src_vocab == tgt_vocab.
    """

    def call(self, src_ids, tgt_ids, training=False,
             enc_mask=None, dec_mask=None):
        if enc_mask is None:
            enc_mask = create_padding_mask(src_ids)
        if dec_mask is None:
            dec_mask = create_decoder_mask(tgt_ids, tgt_len=tf.shape(tgt_ids)[1])
        enc_output, enc_attn = self.encoder(src_ids, mask=enc_mask, training=training)
        dec_output, dec_attn = self.decoder(tgt_ids, enc_output,
                                             dec_mask=dec_mask, enc_mask=enc_mask,
                                             training=training)
        embedding_weights = self.decoder.embedding.embeddings
        logits = tf.matmul(dec_output, embedding_weights, transpose_b=True)
        return logits, {**enc_attn, **dec_attn}
'''



In [52]:

with open(TRANSFORMER_PY, 'w', encoding='utf-8') as f:
    f.write(code)


In [54]:
# Reload and verify
import importlib
import transformer_model
importlib.reload(transformer_model)
from transformer_model import TiedTransformer as TF_loaded

tf_reload = TF_loaded(
    num_layers=2, d_model=512, num_heads=8,
    d_ff=2048, src_vocab=8000, tgt_vocab=8000, dropout=0.0,
)

src_r = tf.constant([[5, 12, 8, 33, 7, 0, 0]], dtype=tf.int32)
tgt_r = tf.constant([[1, 15, 22, 8, 2, 0, 0]], dtype=tf.int32)

logits_r, attn_r = tf_reload(src_r, tgt_r, training=False)

assert logits_r.shape == (1, 7, 8000)

print("transformer_model.py saved and reloaded.")
print(f"path : {TRANSFORMER_PY}")
print(f"logits shape : {logits_r.shape}  (batch, tgt_len, vocab_size)")
print(f"attn entries : {len(attn_r)}")

transformer_model.py saved and reloaded.
path : /content/drive/MyDrive/Transformer_Mine/transformer_model.py
logits shape : (1, 7, 8000)  (batch, tgt_len, vocab_size)
attn entries : 6


/usr/local/lib/python3.13/dist-packages/keras/src/layers/layer.py:424: UserWarning: `build()` was called on layer 'tied_transformer_7', however the layer does not have a `build()` method implemented and it looks like it has unbuilt state. This will cause the layer to be marked as built, despite not being actually built, which may cause failures down the line. Make sure to implement a proper `build()` method.
  warnings.warn(
